# 문이 쾅 닫힐 때의 회전·마찰 동역학 — 데이터 분석 & 시뮬레이션
**Team solvE · Group 14 · DGIST General Physics**

논문: P. Klein *et al.*, *Rotational and frictional dynamics of the slamming of a door*, **Am. J. Phys. 85, 30 (2017)**

스마트폰 **자이로스코프**(Phyphox)로 측정한 $\omega(t)$ 에 6개 마찰 모델을 적합하고,
운동방정식을 수치적분해 검증한다. 그림은 물리 저널 스타일(SciencePlots), 라벨은 영어.

> **방법론:** 논문은 *"smallest SSE is not preferred... evaluated with respect to
> scientific appropriateness"* — 모델 선택은 적합도가 아니라 **계수의 물리적 타당성**으로.

In [1]:
import sys; sys.path.insert(0, "src")
import numpy as np, pandas as pd
import analysis as A
from analysis import (load_segment, fit_all, simulate, CASES, MODELS,
                      MODEL_LABEL, PALETTE, I_DOOR)
from friction_models import MODELS as M
import matplotlib.pyplot as plt
print(f"I = 1/3 m w^2 = {I_DOOR:.4f} kg m^2")

/Users/cew/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


I = 1/3 m w^2 = 3.1688 kg m^2


## 1. 운동방정식
$$I=\tfrac13 mw^2,\qquad \frac{d\omega}{dt}=-(A+B\omega+C\omega^2),\quad A=\tfrac aI,B=\tfrac bI,C=\tfrac cI.$$
구현은 `src/friction_models.py`.

## 2. 자유회전(phase 3) 구간
전체 신호 = 미는 가속 + 자유회전 감속 + 충돌. phase 3 만 적합. 프레임 근접 직전
$|d\omega/dt|$ 가 커지는 air-cushion 구간(phase 4)은 제외(논문과 동일).

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(9,3.2))
for ax,(name,cfg) in zip(axes, CASES.items()):
    d = pd.read_csv(cfg["csv"]); t,w = d["Time (s)"].values, d["Absolute (rad/s)"].values
    ax.plot(t,w,lw=.4,color="#888"); ax.axvspan(*cfg["win"], color=cfg["color"], alpha=.18)
    ax.set_title(cfg["label"]); ax.set_xlabel("t (s)"); ax.set_ylabel(r"$\omega$ (rad/s)")
plt.tight_layout(); plt.show()

/var/folders/v_/4schpzdx1jzcrlr490h3q17h0000gn/T/ipykernel_78264/4087989431.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. 6개 모델 적합 + 적합도

In [3]:
out = A.run()
for name, cd in out["cases"].items():
    print(f"\n=== {cd['label']}  (window {cd['win']}, n={cd['n']}, w {cd['w0_obs']:.2f}->{cd['w_end']:.2f}) ===")
    print(f"{'model':5s} {'R2':>8s} {'SSE':>10s} {'dAIC':>8s}")
    amin = min(cd['fits'][k]['AIC'] for k in MODELS)
    for k in MODELS:
        f = cd['fits'][k]
        print(f"{k:5s} {f['R2']:8.4f} {f['SSE']:10.4g} {f['AIC']-amin:8.1f}")


=== Fast slam  (window (7.69, 8.0), n=144, w 2.08->1.79) ===
model       R2        SSE     dAIC
D       0.9441    0.08098      0.0
S       0.9343     0.0952     23.3
N       0.9238     0.1104     44.7
DS      0.9441    0.08098      2.0
DN      0.9441    0.08098      2.0
SN      0.9343     0.0952     25.3

=== Slow slam  (window (8.9, 11.5), n=1206, w 0.60->0.27) ===
model       R2        SSE     dAIC
D       0.9879       0.12   1785.3
S       0.9967    0.03321    236.5
N       0.9896      0.103   1601.9
DS      0.9967    0.03321    238.5
DN      0.9973    0.02725      0.0
SN      0.9967    0.03277    222.4


In [4]:
for name,cfg in CASES.items():
    t,w = load_segment(cfg["csv"], cfg["win"]); res,best = fit_all(t,w)
    tt = np.linspace(0,t.max(),400)
    plt.figure(figsize=(5,3.4)); plt.scatter(t,w,s=5,color="#444",alpha=.3,label="measured")
    for k in MODELS:
        r=res[k]; fn=M[k]["fn"]; p=[r["params"]["w0"]]+[r["params"][c] for c in M[k]["coeffs"]]
        plt.plot(tt,fn(tt,*p),lw=1.8 if k==best else 1.0,ls="-" if k==best else "--",
                 color=PALETTE[k],label=f"{MODEL_LABEL[k]} ($R^2$={r['R2']:.3f})")
    plt.title(cfg["label"]); plt.xlabel("t (s)"); plt.ylabel(r"$\omega$ (rad/s)")
    plt.legend(fontsize=6); plt.tight_layout(); plt.show()

/var/folders/v_/4schpzdx1jzcrlr490h3q17h0000gn/T/ipykernel_78264/1368270095.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(fontsize=6); plt.tight_layout(); plt.show()


/var/folders/v_/4schpzdx1jzcrlr490h3q17h0000gn/T/ipykernel_78264/1368270095.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(fontsize=6); plt.tight_layout(); plt.show()


## 4. 물리적 타당성 — 모델 선택의 결정 기준
적합 계수를 차원분석 추정과 대조 (논문 식 9·10·11).

In [5]:
pv = out["physical_validity"]; th = pv["theory"]
print(f"estimate: Stokes b/I ~ {th['bI_Stokes']:.1e},  Newton c/I ~ {th['cI_Newton']:.2f}")
for n,c in pv["cases"].items():
    print(f"\n[{n}]  a/I={c['aI']:.3f}  b/I={c['bI']:.3f}  c/I={c['cI']:.3f}")
    print(f"   Dry   : implied mu={c['mu_implied']:.4f}")
    print(f"   Stokes: b/I is {c['bI_ratio']:.0e}x estimate  -> rejected")
    print(f"   Newton: c/I is {c['cI_ratio']:.1f}x estimate (same order) -> accepted")
print(f"\nDry implied-mu inconsistency: fast vs slow = {pv['mu_inconsistency']:.1f}x  -> Dry rejected")
print("F-tests (nested):", out["f_tests"])

estimate: Stokes b/I ~ 5.8e-05,  Newton c/I ~ 0.30

[fast]  a/I=1.088  b/I=0.543  c/I=0.271
   Dry   : implied mu=0.0308
   Stokes: b/I is 9e+03x estimate  -> rejected
   Newton: c/I is 0.9x estimate (same order) -> accepted

[slow]  a/I=0.120  b/I=0.291  c/I=0.675
   Dry   : implied mu=0.0034
   Stokes: b/I is 5e+03x estimate  -> rejected
   Newton: c/I is 2.2x estimate (same order) -> accepted

Dry implied-mu inconsistency: fast vs slow = 9.0x  -> Dry rejected
F-tests (nested): {'fast': {'D_to_DN': 7.249480131752762e-14, 'S_to_SN': 1.8498659367669453e-12, 'N_to_DN': 51.279690454458525}, 'slow': {'D_to_DN': 4092.4922274910737, 'S_to_SN': 16.134215660161217, 'N_to_DN': 3345.196615850278}}


## 5. 수치 시뮬레이션 — 해석해 검증

In [6]:
name="slow"; cfg=CASES[name]; cd=out["cases"][name]; best=cd["best"]; p=cd["fits"][best]["params"]
t,w = load_segment(cfg["csv"], cfg["win"]); tt=np.linspace(0,t.max(),500)
fn=M[best]["fn"]; analytic=fn(tt,*([p["w0"]]+[p[c] for c in M[best]["coeffs"]]))
numeric=simulate(tt,p["w0"],A=p.get("A",0),B=p.get("B",0),C=p.get("C",0))
plt.figure(figsize=(5,3.4)); plt.scatter(t,w,s=5,color="#bbb",alpha=.4,label="measured")
plt.plot(tt,analytic,lw=1.8,color="#1f5e5b",label=f"analytic ({MODEL_LABEL[best]})")
plt.plot(tt,numeric,lw=1.1,ls=":",color="#d1495b",label="numerical (solve_ivp)")
plt.legend(); plt.xlabel("t (s)"); plt.ylabel(r"$\omega$ (rad/s)"); plt.tight_layout(); plt.show()
print("max difference:", np.max(np.abs(analytic-numeric)), "rad/s")

max difference: 9.922413557461596e-09 rad/s


/var/folders/v_/4schpzdx1jzcrlr490h3q17h0000gn/T/ipykernel_78264/739051745.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.xlabel("t (s)"); plt.ylabel(r"$\omega$ (rad/s)"); plt.tight_layout(); plt.show()


## 6. 결론
- **Stokes 기각**: 적합 $b/I$ 가 이론값의 $10^3$–$10^4$배.
- **Dry 기각**: 함의 $\mu$ 가 두 슬램에서 9배 불일치 (재료상수가 변할 수 없음).
- **Newton 채택**: 적합 $c/I$(0.27 / 0.68)가 추정($\sim0.30$)과 같은 자릿수, 저속에서
  커지는 경향까지 논문(Reynolds 의존 $C_D$)과 일치 → **논문과 일치**.
- fast 슬램은 구간이 짧아 통계 구별 불가(논문도 지적) → 물리적 타당성으로 판정.
- 수치적분 vs 해석해 $\sim10^{-8}$rad/s 일치.